<a href="https://colab.research.google.com/github/AsmaAssa2471/my-flyrank-ml-internship/blob/main/work/03_model_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install duckdb huggingface_hub pandas scikit-learn

In [14]:
from huggingface_hub import HfApi, login
from google.colab import userdata

# Token authenticate karna
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

# Repo ki saari files ke paths print karna
api = HfApi()
files = api.list_repo_files(repo_id="FlyRank/internship-warehouse", repo_type="dataset")

print("📁 Repo ke andar yeh paths hain:")
for f in files:
    print(f)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


📁 Repo ke andar yeh paths hain:
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fac

In [15]:
import duckdb
import os
from google.colab import userdata
from huggingface_hub import login

# 1. Token Setup
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN

# 2. DuckDB Connection
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")

# 3. Exact Path Query on dim_content.parquet
try:
    query = """
    SELECT *
    FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    LIMIT 5
    """
    df = con.execute(query).df()
    print("✅ Data Access Successful! Sample Content Data:")
    display(df)
except Exception as e:
    print("❌ Error loading data:", e)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Data Access Successful! Sample Content Data:


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682,2555,NaT,NaT,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438,2430,NaT,NaT,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576,2645,NaT,NaT,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457,2522,NaT,NaT,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776,2552,NaT,NaT,True,False


In [16]:
import duckdb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from google.colab import userdata

# 1. Setup DuckDB Connection
HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")

# 2. Join Tables & Extract Features (Corrected Column Names)
query = """
SELECT
    c.content_hash_id,
    c.word_count,
    c.keyword_char_count,
    c.category_count,
    COALESCE(p.impressions, 0) AS impressions,
    COALESCE(p.clicks, 0) AS clicks,
    CASE
        WHEN COALESCE(p.impressions, 0) > 0 THEN (COALESCE(p.clicks, 0) * 1.0 / p.impressions)
        ELSE 0
    END AS ctr
FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' c
LEFT JOIN (
    SELECT
        content_hash_id,
        SUM(impressions_90d) AS impressions,
        SUM(clicks_90d) AS clicks
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet'
    GROUP BY content_hash_id
) p ON c.content_hash_id = p.content_hash_id
WHERE c.is_published = True AND c.is_deleted = False
LIMIT 10000;
"""

print("⏳ Querying data from warehouse...")
df = con.execute(query).df()
df.fillna(0, inplace=True)

# 3. Model Preparation
X = df[['word_count', 'keyword_char_count', 'category_count', 'impressions']]
y = df['ctr']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Train Random Forest Model
print("🤖 Training Machine Learning Model...")
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 5. Evaluate Model
predictions = model.predict(X_test)
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("\n✅ Model Training Completed Successfully!")
print(f"📊 Mean Squared Error (MSE): {mse:.6f}")
print(f"📈 R2 Score: {r2:.4f}")

# Sample Opportunity Scoring
df['predicted_ctr'] = model.predict(X)
df['opportunity_score'] = df['impressions'] * (df['predicted_ctr'] - df['ctr'])
top_opportunities = df.sort_values(by='opportunity_score', ascending=False).head(5)

print("\n🎯 Top Opportunity Pages (High Impressions, Low Current CTR):")
display(top_opportunities[['content_hash_id', 'impressions', 'clicks', 'ctr', 'predicted_ctr', 'opportunity_score']])

⏳ Querying data from warehouse...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

🤖 Training Machine Learning Model...

✅ Model Training Completed Successfully!
📊 Mean Squared Error (MSE): 0.000058
📈 R2 Score: -0.0613

🎯 Top Opportunity Pages (High Impressions, Low Current CTR):


,content_hash_id,impressions,clicks,ctr,predicted_ctr,opportunity_score
7051,content_3b6e4c8d9a0a5c9c,92045.0,180.0,0.001956,0.004240,210.305267
4378,content_bddfdd871aa09fbe,158082.0,9.0,0.000057,0.001289,194.838024
3954,content_44f34c0a90047651,193482.0,19.0,0.000098,0.001038,181.841766
6968,content_3961d57e9252a4ab,45612.0,3.0,0.000066,0.003703,165.917161
6964,content_39584991d1c2b7a0,93941.0,30.0,0.000319,0.002048,162.346487
